# Custom Energy Parametrization
Demonstrate MeshFEM's custom energy definition.

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'
# os.environ['VECLIB_MAXIMUM_THREADS'] = '1'

import sys
sys.path.append('../')
import MeshFEM
import mesh, py_newton_optimizer, viewer, mesh_energy
import param_utils, benchmark
import numpy as np


In [ ]:
from Benchmark import helper_funcs

In [ ]:
import continuation_parametrization
from continuation_parametrization import parametrization, energy

### Read Mesh

In [ ]:
# m = mesh.Mesh('../../models/hand.msh')

In [ ]:
mesh_path = '../../../Models/PPdata/data1/fig12_a.obj'

In [ ]:
m = helper_funcs.read_mesh(mesh_path)

### initialization

In [ ]:
uv = mesh_energy.NodalVars(m, 2)

In [ ]:
# Scale so that the surface area is pi (to match [Su et al. 2020])
# m.setVertices(m.vertices() * np.sqrt(np.pi / m.volume))
# uv.setVars(param_utils.tutteInitialization(m).ravel())

In [ ]:
bdry_uv = helper_funcs.getBDdataOnNormalizedCircle(m)
uv_init = helper_funcs.tutteInitialization(m, bdry_uv)

In [ ]:
uv_init

In [ ]:
uv.setVars(uv_init.ravel())

## Test modifiedSD

In [ ]:
p_values = np.ones(500)
p_values = np.loadtxt('cm_only_true_area/fig12_a_pp_t.txt')
p_values = np.asarray(p_values, dtype=float)

if p_values.ndim != 1:
    raise ValueError('p_values must be a 1-d array.')
if np.any((p_values < 0.0) | (p_values > 1.0)):
    raise ValueError('AttenuatedSymmetricDirichlet p values must be in [0, 1].')

print(p_values)

In [ ]:
# e = energy.SymmetricARAP(2)

In [ ]:
import flip_avoiding_step_length

def make_problem(p):
    e = energy.AttenuatedSymmetricDirichlet(2, float(p))
    param = parametrization.Parametrization(m, uv, e)
    param.elementHessianShift = 1e-6
    prob = py_newton_optimizer.NewtonMultiobjectiveProblem(uv, [param])
    # flip-avoiding needed ?
    prob.initialFeasibleStepLengthComputer = flip_avoiding_step_length.FlipAvoidingStepLength(m.elements())
    prob.initialFeasibleStepLengthComputer.backoffFactor = 0.8
    
    prob.hessianShift = 0 # Work around energy nullspace by adding a small shift
    # prob.useRelativeHessianShift = True
    return e, param, prob

def eval_prob():
    e_eval = energy.AttenuatedSymmetricDirichlet(2, 1.0)
    param_eval = parametrization.Parametrization(m, uv, e_eval)
    prob_eval = py_newton_optimizer.NewtonMultiobjectiveProblem(uv, [param_eval])
    
    energy_eval = prob_eval.energy()
    grad_norm_eval = np.linalg.norm(prob_eval.gradient())
    return energy_eval, grad_norm_eval

In [ ]:
e, param, prob = make_problem(p_values[0])
print(e.p)

In [ ]:
grad_tol = 2e-8

In [ ]:
def make_optimizer(prob, niter=1):
    opt = prob.optimizer()
    opt.options.niter = niter
    opt.options.gradTol = grad_tol
    opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionAlways()
    return opt

In [ ]:
em = MeshFEM.EmbeddedMesh(m, uv)
v = viewer.Viewer(em, wireframe=True)
v.show()

In [ ]:
opt = make_optimizer(prob, niter=1)
# prob.setCustomIterationCallback(v.updater())

In [ ]:
benchmark.reset()
reports = []
p_trace = []

for it, p in enumerate(p_values):
    e, param, prob = make_problem(p)
    opt = make_optimizer(prob, niter=1)
    rep = opt.optimize()
    reports.append(rep)
    energy_eval, grad_norm_eval = eval_prob()
    
    p_trace.append({
        'iter': it,
        'p': float(e.p),
        'energy': energy_eval,
        'grad_norm': grad_norm_eval,
    })
    
    # if grad_norm < grad_tol: break

benchmark.report()
v.update()

## Post Opt

In [ ]:
def debugPtrace(p_trace):
    for p_t in p_trace:
        print(f"{p_t['iter']}    {p_t['p']}    {p_t['energy']}    {p_t['grad_norm']}")

In [ ]:
debugPtrace(p_trace)

In [ ]:
len(p_trace)